In [1]:
import os
import cx_Oracle
import pandas as pd
from tqdm import tqdm
import time
from collections import Counter

In [2]:
print(os.listdir(r"C:\Wallet_MangadexDatabase"))

['cwallet.sso', 'ewallet.p12', 'ewallet.pem', 'keystore.jks', 'ojdbc.properties', 'README', 'sqlnet.ora', 'tnsnames.ora', 'truststore.jks']


In [3]:
os.environ["TNS_ADMIN"] = r"C:\Wallet_MangadexDatabase"

In [4]:
df = pd.read_csv("mangadex_final.csv")

In [5]:
df.fillna({"last_chapter": 0, "year": 0, "demographic": "Unknown"}, inplace=True)

In [6]:
username = ""
password = ""
dsn = "mangadexdatabase_high"  

conn = cx_Oracle.connect(user=username, password=password, dsn=dsn)
cursor = conn.cursor()

In [10]:
df = pd.read_csv("mangadex_final.csv")
df.fillna({
    'title': '',
    'description': '',
    'status': '',
    'originalLanguage': '',
    'publicationDemographic': ''
}, inplace=True)
df['lastChapter'] = pd.to_numeric(df['lastChapter'], errors='coerce')

df['status'] = df['status'].str.strip().str.lower()
df['originalLanguage'] = df['originalLanguage'].str.strip().str.lower()

# 1. Insert to dim_manga
manga_data = [
    (
        str(row['id']),
        str(row['title'])[:255],
        str(row['description'])
    )
    for _, row in df.iterrows()
]

batch_size = 500
merge_sql = """
MERGE INTO dim_manga d
USING (SELECT :1 AS manga_id, :2 AS title, :3 AS description FROM dual) s
ON (d.manga_id = s.manga_id)
WHEN NOT MATCHED THEN
    INSERT (manga_id, title, description)
    VALUES (s.manga_id, s.title, s.description)
"""

for i in range(0, len(manga_data), batch_size):
    cursor.executemany(merge_sql, manga_data[i:i + batch_size])
    conn.commit()

# 2. Insert to dim_demographic
unique_demos = df['publicationDemographic'].dropna().unique()
cursor.executemany("""
    MERGE INTO dim_demographic d
    USING (SELECT :1 AS demographic FROM dual) s
    ON (d.demographic = s.demographic)
    WHEN NOT MATCHED THEN INSERT (demographic) VALUES (:1)
""", [(demo,) for demo in unique_demos])
conn.commit()

cursor.execute("SELECT demographic, demographic_id FROM dim_demographic")
demo_cache = {demo.strip(): demo_id for demo, demo_id in cursor.fetchall()}

# 3. Insert to dim_time
unique_years = sorted(df['year'].dropna().unique())
unique_years = [(int(year),) for year in unique_years if pd.notna(year)]

cursor.executemany("""
    MERGE INTO dim_time t
    USING (SELECT :1 AS year FROM dual) s
    ON (t.year = s.year)
    WHEN NOT MATCHED THEN INSERT (year) VALUES (s.year)
""", unique_years)
conn.commit()

# 4. Insert to dim_language
unique_langs = df['originalLanguage'].dropna().unique()
cursor.executemany("""
    MERGE INTO dim_language l
    USING (SELECT :1 AS original_language FROM dual) s
    ON (l.original_language = s.original_language)
    WHEN NOT MATCHED THEN INSERT (original_language) VALUES (:1)
""", [(lang,) for lang in unique_langs])
conn.commit()

# 5. Insert to dim_status
unique_statuses = sorted(set(s.strip().lower() for s in df['status'].dropna() if s.strip()))
cursor.executemany("""
    MERGE INTO dim_status s
    USING (SELECT :1 AS status FROM dual) src
    ON (LOWER(TRIM(s.status)) = LOWER(TRIM(src.status)))
    WHEN NOT MATCHED THEN INSERT (status) VALUES (:1)
""", [(status,) for status in unique_statuses])
conn.commit()

# Caches
cursor.execute("SELECT language_id, original_language FROM dim_language")
language_cache = {lang.strip().lower(): lang_id for lang_id, lang in cursor.fetchall()}

cursor.execute("SELECT status_id, status FROM dim_status")
status_cache = {status.strip().lower(): status_id for status_id, status in cursor.fetchall()}

cursor.execute("SELECT manga_id FROM dim_manga")
valid_manga_ids = {row[0] for row in cursor.fetchall()}

cursor.execute("SELECT manga_id, year, demographic_id FROM fact_manga")
existing_keys = set(cursor.fetchall())

# 6. Insert to fact_manga
fact_data = []
fact_seen = set()

print("Preparing fact_manga...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    manga_id = str(row['id'])
    year = row.get('year')
    demo_id = demo_cache.get(row['publicationDemographic'].strip())
    language_id = language_cache.get(row['originalLanguage'])
    status_id = status_cache.get(row['status'])

    if (
    manga_id in valid_manga_ids and pd.notna(year) and
    demo_id is not None and language_id is not None and status_id is not None
    ):
        key = (manga_id, int(year), demo_id)
        if key not in fact_seen and key not in existing_keys:
            lc = row['lastChapter']
            last_chapter = int(float(lc)) if pd.notna(lc) else None
            desc_len = len(str(row['description'])) if pd.notna(row['description']) else 0
            fact_data.append((manga_id, int(year), demo_id, language_id, status_id, last_chapter, desc_len))
            fact_seen.add(key)

if fact_data:
    cursor.executemany("""
        INSERT INTO fact_manga (
            manga_id, year, demographic_id, language_id, status_id, last_chapter, description_length
        ) VALUES (:1, :2, :3, :4, :5, :6, :7)
    """, fact_data)
    conn.commit()
    print(f"Inserted {len(fact_data)} new rows to fact_manga.")
else:
    print("No new rows inserted")

# 7. Remove possible dupes
cursor.execute("""
    DELETE FROM fact_manga mf
    WHERE ROWID NOT IN (
        SELECT MIN(ROWID)
        FROM fact_manga
        GROUP BY manga_id, year, demographic_id
    )
""")
conn.commit()

# Finished
print("\nDONE")
print(f"Inserted: {len(manga_data)} manga, {len(unique_demos)} demographics, {len(fact_data)} facts")


Preparing fact_manga...


100%|█████████████████████████████████████████████████████████████████████████| 22217/22217 [00:01<00:00, 14810.42it/s]


Inserted 21704 new rows to fact_manga.

DONE
Inserted: 22217 manga, 4 demographics, 21704 facts
